<a href="https://colab.research.google.com/github/FelixvanDijk/DL-Lab3/blob/main/lab-3-training/lab-3-training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Techniques for training DNNs


## Contents

1. [Tuning Learning Rate](#Tuning-Learning-Rate)
2. [Tuning Batch Size](#Tuning-Batch-Size)
3. [Batch Normalisation](#Batch-Normalisation)
4. [SGD with Momentum](#SGD-with-Momentum)
5. [Addendum 1: A note on Batch Normalisation](#Addendum-1:-A-note-on-Batch-Normalisation)
6. [Addendum 2: Learning Rate Decay](#Addendum-2:-Learning-Rate-Decay)
7. [Addendum 3: CHALLENGE! Implementing BatchNorm2d](#Addendum-3:-CHALLENGE!-Implementing-BatchNorm2d)
---

**You will be continuing on the code from last lab. You may want to make a separate copy of the code**

Last lab session you built your first CNN and trained it on CIFAR-10. Let's revisit
the loss and accuracy curves we got last time.

![Last lab-session's loss curve](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-2-cnns/media/expected-loss.png?raw=1)

*Legend:*
- Train: Orange
- Test: Red

![Last lab-session's accuracy curve](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-2-cnns/media/expected-accuracy.png?raw=1)

*Legend:*
- Train: Grey
- Test: Blue

What can we learn by looking at these graphs?

1. The **training accuracy** (~70%) could be higher; this dataset has 10 classes, so chance performance would be 10%. *Make sure you distinguish between the training curve and the test/validation curve for both loss and accuracy*. This indicates that our model is underfitting the training data. Our training performance, in normal circumstances, is typically an upper bound of the accuracy we can expect on the test set. In fact, as a rule of thumb, it is a good idea to overfit your training data, then regularise the model to reduce the train-validation gap in order to generalise to future data.

2. The **training loss** decreases quite slowly, perhaps our learning rate is too low.

3. The **test loss** is close to the training loss, this is good, it means that our model generalises well from our training data. Towards the end of training it does seem to overfit a bit, but the **test accuracy** is still increasing, which is our primary goal.


Based on the above, we will first attempt to adjust the learning rate.

**Question:** What are some hyperparameters would you consider changing when observing an overfitting behaviour of your model?

**Question** How could your dataset contribute to an overfitting behaviour of the model?

ANSWER:

When you observe overfitting (where your model performs excellently on the training set but poorly on unseen validation or test data), you can adjust several hyperparameters and look closely at your dataset structure. Here are the answers to your questions:

1. Hyperparameters to Change for Overfitting
Regularisation Strength ($L_1$$L_1$/$L_2$$L_2$ decay): Increasing weight decay (e.g., in your optimizer weight_decay=1e-4) penalizes large weights, forcing the network to learn simpler, smoother decision boundaries.
Dropout Rate: Adding or increasing Dropout layers (e.g., nn.Dropout(p=0.5)) randomly deactivates neurons during training, which prevents neurons from co-adapting too much.
Learning Rate (LR) Schedule: Lowering the learning rate or decaying it over time helps the model fine-tune near a generalizable minimum instead of overshooting and memorizing the training subset.
Batch Size: Modifying batch size (sometimes larger batch sizes have a subtle regularizing effect, though smaller batches introduce noise which can occasionally help escape sharp, overfitted local minima).
Model Capacity (Architecture): Reducing the number of layers, feature channels, or fully connected units directly restricts the capacity of your model to memorize noise.
2. How the Dataset Contributes to Overfitting
Small Dataset Size: If the dataset has too few examples, the neural network can easily "memorize" every single sample and its label instead of learning general features (like edges, shapes, or textures).
Lack of Variety/Diversity: If your training set lacks diverse scenarios (e.g., all dogs are pictured in bright daylight, but test dogs are at night), the model overfits to the context (the daylight background) rather than the object of interest.
Data Leakage: If similar or identical images exist in both your training and validation/test splits (e.g., heavily overlapping video frames), your validation metrics will look great but will fail to generalize to completely new datasets.
Noisy or Incorrect Labels: If a notable percentage of training samples have corrupt labels, a highly expressive network will bend its decision boundaries to memorize those incorrect labels, degrading overall generalization.

---
## Tuning Learning Rate

We've been training our networks with stochastic gradient descent, this updates the parameters $\mathbf{W}$ according to the update rule

$$\mathbf{W}_{t+1} = \mathbf{W}_t - \eta\nabla_{W_t} \mathcal{L}(\mathbf{x}, \mathbf{W}_t)$$

Where
- $\mathbf{x}$ is the mini-batch of inputs to the network.
- $\mathbf{W}_t$ are the network's weights at time $t$.
- $\mathcal{L}$ is the loss function.
- $\nabla_{W_t}\mathcal{L}$ is the matrix containing the partial derivatives of the loss function with respect to the parameters $\mathbf{W}_t$.
- $\eta$ is the learning rate.

When picking a learning rate, it's best to search exponentially varying values, e.g. 0.001, 0.005, 0.01, 0.05, 0.1.

**Task**: The default learning rate in the code is 1e-2, try training with 5e-2 and 1e-1 by running `train_cifar.py` with `--learning-rate <lr>` (replace `<lr>` with `5e-2` or `1e-1`). Which learning rate gives you best results?

We found setting the learning rate to 1e-1 to be the most effective, boosting the test accuracy by ~7%. Let's examine the learning curves this time to see what we can infer from them.

![LR=1e-1 loss curve](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/loss-curve-bs=128-lr=1e-1.png?raw=1) ![LR=1e-1 accuracy curve](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/accuracy-curve-bs=128-lr=1e-1.png?raw=1)

These graphs paint a very different picture to the ones you see at the top. The training accuracy saturates at 100%, the network performs incredibly well on the training set, however the test accuracy plateaus at ~70% and the test loss is increasing. This means our network has *overfitted* to the training data, its performance on the training data isn't reflective of its performance on unseen data.   
We can use regularisation techniques to reduce this effect to try to close the gap between the train/test performance.

Before moving on, for the sake of completeness, we present the results of scanning over a range of learning rate values and the final test set accuracy we achieved.


| Learning rate | Test accuracy (%) |
|---------------|-------------------|
| 1e-4 | 20.52 |
| 5e-4 | 31.35 |
| 1e-3 | 42.32 |
| 5e-3 | 54.47 |
| 1e-2 | 63.40 |
| 5e-2 | 72.04 |
| 1e-1 | 73.39 |
| 2e-1 | 71.19 |
| 5e-1 | 26.66 |

If you were to attempt to reproduce these results, you would not produce the same numbers due to the random initialisation of weights and contents of batches. However you should observe the same trend.

**Q.** Why do you think the network's performance dropped so much when using 5e-1 (0.5) for the learning rate?

ANSWER:
When the learning rate is set too high (such as 0.5), the performance drops drastically due to gradient divergence (or overshooting).

Here is what happens in detail:

Overshooting Minima: The step size taken during weight updates ($W_{t+1} = W_t - \eta\nabla\mathcal{L}$$W_{t+1} = W_t - \eta\nabla\mathcal{L}$) becomes too large. Instead of descending smoothly toward a local minimum on the loss landscape, the updates overshoot it and land on a higher point on the opposite side of the valley.
Divergence: As the steps continue to oscillate wildly with increasing magnitudes, the loss function begins to diverge (increases towards infinity), or the model parameters become extremely large, sometimes resulting in numerical overflow (NaN values).
Saturated Activations: Highly inflated weight values push the inputs of activation functions (such as Sigmoid, Tanh, or even ReLU in terms of dying neurons) into fully saturated regions where gradients become zero or near-zero, rendering any future learning impossible.
Inability to Learn Complex Features: In extreme cases of high learning rates, the model fails to learn anything meaningful and resorts to random guessing, which explains why the accuracy falls back towards ~10% (the baseline chance probability for 10 classes in CIFAR-10).


**Question:** What are some challenges the model could encounter when calculating the gradients? How would you go about mitigating these challenges when designing your model?

### ANSWER:

When calculating gradients in deep neural networks, several critical challenges can arise during backpropagation. Here are the most prominent issues and the methods used to mitigate them:

#### 1. Vanishing Gradients
* **The Challenge:** In very deep networks, as gradients are backpropagated through multiple layers, they are repeatedly multiplied by weights and derivatives of activation functions. If these values are small (e.g., the saturated regions of Sigmoid or Tanh, where the derivative is close to $0$), the gradient shrinks exponentially. By the time it reaches the early layers, it becomes virtually zero, meaning those weights stop updating, and the network ceases to learn.
* **Mitigation:**
  * **Non-saturating Activations:** Use **ReLU** ($f(x) = \max(0, x)$) or its variants like LeakyReLU/ELU instead of Sigmoid/Tanh. ReLU has a constant derivative of $1$ for all positive inputs, preventing gradient decay.
  * **Smart Parameter Initialization:** Use **He (Kaiming) Initialization** (for ReLU) or **Glorot (Xavier) Initialization** (for Sigmoid/Tanh). These methods scale the initial weights based on the layer's fan-in and fan-out, keeping the variance of activations and gradients stable across layers.
  * **Skip Connections (Residual Links):** In architectures like ResNets, shortcut connections let the gradient bypass convolutional blocks and flow directly back without decay ($a^{[l+2]} = g(z^{[l+2]} + a^{[l]})$).

#### 2. Exploding Gradients
* **The Challenge:** Conversely, if the weights are initialized too large or consecutive gradients are repeatedly multiplied by values greater than $1$, the gradients grow exponentially. This leads to unstable, massive parameter updates, causing the loss to fluctuate wildly, diverge to infinity, or output `NaN` values.
* **Mitigation:**
  * **Gradient Clipping:** Forcefully cap the maximum value of the gradient during optimization (e.g., `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`).
  * **Batch Normalization:** Normalizing the inputs to each layer restricts activations to a stable range, which prevents local gradients from blowing up in deeper layers.

#### 3. Internal Covariate Shift
* **The Challenge:** As parameters of previous layers update during training, the distribution of inputs to later layers shifts continuously. This requires subsequent layers to constantly adapt to new distributions, slowing down convergence.
* **Mitigation:**
  * **Batch Normalization (BN):** Normalizing intermediate layers ensures stable zero-mean and unit-variance distributions throughout training, which smoothens the optimization landscape and allows the use of much higher learning rates.

#### 4. Saddle Points and Poor Local Minima
* **The Challenge:** In high-dimensional loss landscapes, optimization can stall in flat regions (saddle points) where gradients are close to $0$ in most directions, making standard SGD extremely slow to escape.
* **Mitigation:**
  * **Momentum-based Optimizers:** Using **Momentum** or adaptive optimizers like **Adam/RMSprop** adds a accumulation vector (velocity) to the updates, allowing the optimizer to push through flat regions and escape shallow plateaus.

---
## Tuning Batch Size

Another fundamental hyperparameter in training is the batch size—i.e. the number of examples that we propagate through the network at each iteration. The risk of smaller batches is that they aren't sufficiently representative of the entire dataset and so the parameter updates (particularly with a large learning rate) overfit that batch. Larger batches with a fixed number of epochs results in fewer weight updates, as the number of iterations in an epoch is reduced.

**Task:** By default the code provided uses a batch size of 128, using your new learning rate (1e-1), try varying the batch-size by specifying the `--batch-size <bs>` argument when running `train_cifar.py`. What happens when you use batches of 64 or 256 images?

Again, for completeness, we provide a comprehensive evaluation of batch sizes when holding the learning rate at 1e-1.


| Batch size | Test accuracy (%) | Number of steps | Time (s) |
|------------|-------------------|-----------------|----------|
| 1          | 10.00             | 1M              | 2366     |
| 2          | 10.00             | 500k            | 1193     |
| 4          | 10.00             | 250k            | 601      |
| 8          | 37.51             | 125k            | 323      |
| 16         | 51.69             | 62.5k           | 288      |
| 32         | 59.08             | 31.25k          | 190      |
| 64         | 72.95             | 15.63k          | 144      |
| 128        | 71.21             | 7.81k           | 116      |
| 256        | 68.78             | 3.91k           | 94       |
| 512        | 68.57             | 1.95k           | 78       |
| 1024       | 67.33             | 0.98k           | 74       |

Note the change in runtime depending on batch size. We can complete the same number of epochs with larger batches quicker than smaller batches as we make better utilisation of the GPU's parallel processing ability.

With a batch size of 4 examples, the network ends up always predicting the same class--this suggests the learning rate is far too high for this number of examples per batch and causes the loss to diverge.
The accuracy peaks at a batch size of 64 and then starts to decrease, this suggests that for the largest batch sizes, there aren't a sufficient number of parameter updates to train the network to convergence. We can validate this hypothesis by comparing the loss curves from the BS=16 experiment and the BS=1024 experiment.


Batch-Size: 64
![Loss curve BS=64](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/loss-curve-bs=64-lr=1e-1.png?raw=1)

Batch-Size:1024
![Loss curve BS=1024](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/loss-curve-bs=1024-lr=1e-1.png?raw=1)

The network with BS=1024 is still training as the training loss hasn't plateaued like in the BS=64 experiment. There is strong overfitting in the BS=64 experiment as you can see an increasing gap between training and test loss. The larger batches have a regularising effect on the training of the network.

**Question** What advantages could shuffling the data  introduce when generating the batches to train your model?

### ANSWER:

Shuffling the training dataset before constructing mini-batches introduces several critical mathematical and practical advantages:

1. **Prevents Order Bias and Catastrophic Forgetting:**
   If the dataset is naturally stored in groups (e.g., all "airplane" images first, then all "automobile" images), training on non-shuffled batches means the model would only see one class for consecutive steps. The network would quickly bias its weights to favor that active class, forgetting the representations it learned from previous classes. Shuffling ensures every mini-batch contains a diverse mix of all target labels.

2. **Ensures Unbiased Gradient Estimates:**
   Stochastic Gradient Descent (SGD) relies on the mathematical principle that the gradient computed from a single mini-batch is an *unbiased estimator* of the gradient over the entire dataset. This assumption only holds true if each mini-batch is a random, independent sample of the total data population.

3. **Regularization and Escaping Local Minima:**
   Shuffling introduces a healthy level of stochastic noise to the parameter updates. This statistical variance helps the optimizer jump out of flat saddle points and shallow, poor local minima on the highly complex loss landscape, guiding the parameters toward flatter, more generalizable valleys.

---

## Batch Normalisation

The distribution of each layer's input, i.e. either the input data itself or the output from the previous hidden layer, plays a crucial role when training. This distribution varies during training. This phenomenon is known as *internal covariate shift*.

It would ease the optimisation of our network if such distributions didn't change during training and adjusting the model's parameters.

This is what batch normalisation does by normalising layer inputs. It was proposed in 2015 by Sergey Ioffe and Christian Szegedy at Google in [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/pdf/1502.03167.pdf). Let's see how it works.

From the paper:

> By fixing the distribution of the layer inputs as the training progresses, we expect to improve the training speed. It has been long known that the network training converges faster if its inputs are whitened – i.e., linearly transformed to have zero means and unit variances, and decorrelated. As each layer observes the inputs produced by the layers below, it would be advantageous to achieve the same whitening of the inputs of each layer.

Consequently they propose normalising the input $x_i$ to each layer on a per channel basis (they skip decorrelating inputs).

$$ \hat{x}_{i, c, x, y} = \frac{x_{i, c, x, y} - \mu_c}{\sigma_c^2 + \epsilon}$$

Where
- $x$ is the output from the previous layer
- $\hat{x}$ is the input to the subsequent layer
- $i, c, x, y$ are the batch, channel, width, and height indices
- The mean $\mu_c$ and variance $\sigma_c^2$ are computed over each training batch
- $\epsilon$ is a small constant for numerical stability.

One issue with batch normalisation is that whilst it speeds up convergence, it restricts the possible functions the neural network can approximate. To restore the representational power of the network, two new parameters per channel are learnt: a multiplier $\gamma_c$ and a bias $\beta_c$ to adapt the mean and variance of the normalised input. Refer to your lectures for more details.

$$ \hat{x}_{i, c, x, y} = \gamma_c \frac{x_{i, c, x, y} - \mu_c}{\sqrt{\sigma_c^2 + \epsilon}} + \beta_c $$

Batch normalisation is typically added **after** each convolutional or fully connected layer **before** the activation function.


**Task**:
- Add [`BatchNorm2d`](https://pytorch.org/docs/1.2.0/nn.html#batchnorm2d) layers after each convolutional and [`BatchNorm1d`](https://pytorch.org/docs/1.2.0/nn.html#batchnorm1d) after your first fully connected layer (you wouldn't put it on your output fully connected layer).
- Remember you should define the layers in the `__init__` method of your `CNN` class and then call it in the `forward` pass after performing the convolution and before the activation function. Each BatchNorm layer will have different input dimensions and will need to learn different parameters, so it is important that you define separate BatchNorm layers in your `__init__` method.
- Change `tb_log_dir_prefix` in `get_summary_writer_log_dir` to read `f'CNN_bn_bs={args.batch_size}_lr={args.learning_rate}_run_'` (this way we can distinguish our logs from the batch normalised networks and the un-normalised networks).
- Train your network with BS=128 and LR=1e-1. Expect to get an accuracy of ~74%.

If we compare the loss of the batch-normalized network and the previous version without batch-normalization, we can see that it converges much faster, yet interestingly it also does not overfit as much, it has a reguarlising effect.

![BN vs. Non-BN network loss](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/loss-bn-vs-non-bn.png?raw=1)

BN network performs better (~2%) in terms of accuracy.

![BN vs. Non-BN network accuracy](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/accuracy-bn-vs-non-bn.png?raw=1)

**NOTE:** Recent research casts doubt on the reason for batch normalisation's success, see [Addendum 1](#Addendum-1:-A-note-on-Batch-Normalisation) for more information on this (*optional*).

**Question** How does batch norm work during inference?

ANSWER:

During inference (evaluation mode, when `model.eval()` is called), Batch Normalization behaves differently than during training to ensure that the model produces deterministic, consistent predictions regardless of the inference batch size (even if predicting on a single image).

Here is how it works under the hood:

1. **Using Running (Global) Statistics:**
   Instead of calculating the mean ($

u$) and variance ($
	sigma^2$) over the current mini-batch, inference uses the **running mean** and **running variance** that were accumulated and tracked during the training phase. These are stored as non-trainable buffers (`running_mean` and `running_var`) within the PyTorch layer.

2. **The Inference Normalization Equation:**
   For a test sample channel $x$, the normalization is computed as:
   $$\hat{x} = \frac{x - \mu_{\text{running}}}{\sqrt{\sigma^2_{\text{running}} + \epsilon}}$$
   This step is completely deterministic and does not depend on other samples in the inference batch.

3. **Applying Learned Affine Parameters:**
   After normalization, the scale ($\gamma$) and shift ($\beta$) parameters learned during training are applied:
   $$y = \gamma \hat{x} + \beta$$

4. **Linear Simplification (Folded BatchNorm):**
   Because all parameters ($
	um_{\text{running}}$, $
	sigma^2_{\text{running}}$, $\gamma$, $\beta$) are static during inference, the entire operation becomes a simple linear transformation. In production environments, this allows the Batch Normalization layer to be mathematically "folded" directly into the preceding convolutional or linear layer's weights and biases, speeding up inference time with zero loss in accuracy.

---
## SGD with Momentum

A useful technique using SGD is the concept of momentum.
We can think of the surface of the loss function we're optimising as a landscape.
Each point on that landscape corresponds to a set of network parameters.
In SGD, we compute the steepest direction down from our current position and take a step in that direction iteratively until we reach one of the minima.
An alternative approach is to simulate a physical ball rolling across this landscape.
As the ball rolls down the hill, it picks up momentum, it is not only the steepest direction that defines the ball's trajectory, but also its current velocity.
We can do exactly the same thing with our network during the optimisation process.
We give our ball (representing the network parameters) an initial velocity of 0, then compute its acceleration as a step in the steepest direction down.
We update the ball's velocity according to its previous velocity and its current acceleration and then update its position.

Practically, let's compare the implementation of these two strategies:

Without momentum, i.e. normal SGD, we compute the steepest direction, and take a step of size $\eta$ (the learning rate) in the opposite direction

$$\mathbf{W}_{t+1} = \mathbf{W}_t - \eta\nabla_{W_t} \mathcal{L}(\mathbf{x}, \mathbf{W}_t)$$

With momentum, we simulate a ball moving in the loss' landscape. It starts from being stationary, i.e. it's velocity is 0.

$$ v_0 = 0 $$

We then update it's velocity according to its acceleration, the gradient of the surface (akin to gravity).

$$ v_{t + 1} = \mu v_t - \eta\nabla_{W_t} \mathcal{L}(\mathbf{x}, \mathbf{W}_t) $$

where

- $v_t$ represents the ball's velocity at time $t$
- $\mu$ is the *momentum* coefficient, typically set to somewhere around 0.9.
  When $\mu = 0$ the ball has no momentum (like in normal SGD) and when $\mu = 1$ the particle does not suffer friction.

We update the ball's position according to its velocity $v_t$

$$ W_{t + 1} = W_t + v_t $$



**Task:**
1. Locate the part of your code defining the SGD optimizer
   ```python
       optimizer = optim.SGD(model.parameters(), args.learning_rate)
   ```
2. Add an argument `momentum` to your [`SGD`](https://pytorch.org/docs/1.2.0/optim.html#torch.optim.SGD) optimizer, and set that to 0.9.
3. Change `tb_log_dir_prefix` in `get_summary_writer_log_dir` to read
   `f'CNN_bn_bs={args.batch_size}_lr={args.learning_rate}_momentum=0.9_run_'`.
4. Train the network with a batch size of 128, learning-rate of 1e-1. Expect to get a test accuracy of around 77%.

**NOTE** You could instead define an argument at the top of your file
```python
    parser.add_argument("--sgd-momentum", default=0, type=float)
```
and pass this into the `SGD` constructor invocation. This allows you to change the momentum value (0.9) from the command line, similar to how you can control the learning rate and batch size.

If we compare our loss and accuracy curves with and without momentum we can see that with momentum the network converges a little quicker and reaches a higher test accuracy.

![Loss curves: momentum vs. no momentum](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/loss-momentum-vs-no-momentum.png?raw=1)
![Accuracy curves: momentum vs. no momentum](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/accuracy-momentum-vs-no-momentum.png?raw=1)

ANSWER:

**Yes, momentum can hurt training under certain conditions.** While it generally accelerates convergence and helps navigate flat regions, using it incorrectly or with poorly tuned hyperparameters can cause several issues:

1. **Overshooting Minima (Oscillations):**
   If the momentum coefficient ($

um$) is too high or the learning rate is too large, the accumulated velocity can cause the optimizer to overshoot deep, narrow local minima or even the global minimum. The ball carries too much kinetic energy to settle, leading to severe oscillations around stable valleys and slow convergence near the end of training.

2. **Exacerbating Gradient Explosion:**
   In regions with extremely steep gradients, standard SGD might make a single excessively large step. With momentum, that massive step creates a high velocity vector that is carried over into subsequent updates, compounding the error and rapidly driving the model toward numerical instability (`NaN` losses).

3. **False Saturation (Cruising past complex boundaries):**
   In high-dimensional space, the optimization landscape may require a sharp change of direction to fit complex, delicate boundaries. High momentum makes the parameter updates less sensitive to instantaneous local gradients, forcing the optimizer to cruise straight through subtle, optimal regions because of its past direction history.

### Mitigation
To counteract these side effects, researchers often use **Nesterov Accelerated Gradient (NAG)** momentum, which computes the gradient step *after* applying the current velocity (looking ahead), or use an **Adaptive Optimizer (like Adam or RMSprop)** which dynamically scales updates on a per-parameter basis rather than relying on a global momentum constant.

In [17]:
#!/usr/bin/env python3
import time
from multiprocessing import cpu_count
from typing import Union, NamedTuple

import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

import argparse
from pathlib import Path

torch.backends.cudnn.benchmark = True

parser = argparse.ArgumentParser(
    description="Train a simple CNN on CIFAR-10",
    formatter_class=argparse.ArgumentDefaultsHelpFormatter,
)
default_dataset_dir = Path.home() / ".cache" / "torch" / "datasets"
parser.add_argument("--dataset-root", default=default_dataset_dir)
parser.add_argument("--log-dir", default=Path("logs"), type=Path)
parser.add_argument("--learning-rate", default=1e-1, type=float, help="Learning rate")
parser.add_argument(
    "--batch-size",
    default=128,
    type=int,
    help="Number of images within each mini-batch",
)
parser.add_argument(
    "--epochs",
    default=20,
    type=int,
    help="Number of epochs (passes through the entire dataset) to train for",
)
parser.add_argument(
    "--val-frequency",
    default=2,
    type=int,
    help="How frequently to test the model on the validation set in number of epochs",
)
parser.add_argument(
    "--log-frequency",
    default=10,
    type=int,
    help="How frequently to save logs to tensorboard in number of steps",
)
parser.add_argument(
    "--print-frequency",
    default=10,
    type=int,
    help="How frequently to print progress to the command line in number of steps",
)
parser.add_argument(
    "-j",
    "--worker-count",
    default=cpu_count(),
    type=int,
    help="Number of worker processes used to load data.",
)

class ImageShape(NamedTuple):
    height: int
    width: int
    channels: int

class CNN(nn.Module):
    def __init__(
        self,
        height: int = 32,
        width: int = 32,
        channels: int = 3,
        class_count: int = 10,
    ):
        super().__init__()
        self.input_shape = ImageShape(height, width, channels)
        self.class_count = class_count

        self.conv1 = nn.Conv2d(
            in_channels=self.input_shape.channels,
            out_channels=32,
            kernel_size=(5, 5),
            padding=(2, 2),
        )
        self.initialise_layer(self.conv1)

        # BatchNorm2d for first convolutional layer (32 output channels)
        self.bn1 = nn.BatchNorm2d(num_features=32)

        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=(5, 5),
            padding=(2, 2),
        )
        self.initialise_layer(self.conv2)

        # BatchNorm2d for second convolutional layer (64 output channels)
        self.bn2 = nn.BatchNorm2d(num_features=64)

        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

        self.fc1 = nn.Linear(4096, 1024)
        self.initialise_layer(self.fc1)

        # BatchNorm1d after the first fully connected layer (1024 output units)
        self.bn_fc1 = nn.BatchNorm1d(num_features=1024)

        self.fc2 = nn.Linear(1024, self.class_count)
        self.initialise_layer(self.fc2)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(images)))
        x = self.pool1(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = torch.flatten(x, start_dim=1)

        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.fc2(x)

        return x

    @staticmethod
    def initialise_layer(layer):
        if hasattr(layer, "bias") and layer.bias is not None:
            nn.init.zeros_(layer.bias)
        if hasattr(layer, "weight") and layer.weight is not None:
            nn.init.kaiming_normal_(layer.weight)

class Trainer:
    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        criterion: nn.Module,
        optimizer: Optimizer,
        summary_writer: SummaryWriter,
        device: torch.device,
    ):
        self.model = model.to(device)
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.summary_writer = summary_writer
        self.step = 0

    def train(
        self,
        epochs: int,
        val_frequency: int,
        print_frequency: int,
        log_frequency: int,
    ):
        self.model.train()
        for epoch in range(epochs):
            self.model.train()
            data_load_start_time = time.time()
            for batch, labels in self.train_loader:
                batch = batch.to(self.device)
                labels = labels.to(self.device)
                data_load_time = time.time() - data_load_start_time
                step_start_time = time.time()

                logits = self.model.forward(batch)
                loss = self.criterion(logits, labels)

                loss.backward()

                self.optimizer.step()
                self.optimizer.zero_grad()

                self.step += 1
                step_time = time.time() - step_start_time

                if self.step % print_frequency == 0:
                    try:
                        preds = logits.argmax(dim=-1)
                        accuracy = compute_accuracy(labels, preds)
                    except NameError:
                        accuracy = 0.0

                    print(
                        f"epoch: [{epoch}], step: [{self.step}], batch loss: {loss.item():.5f}, "
                        f"batch accuracy: {accuracy * 100:2.2f}, data load time: {data_load_time:.5f}, "
                        f"step time: {step_time:.5f}"
                    )

                if self.step % log_frequency == 0:
                    self.summary_writer.add_scalar("loss/train", loss.item(), self.step)
                    try:
                        self.summary_writer.add_scalar("accuracy/train", accuracy, self.step)
                    except NameError:
                        pass

                data_load_start_time = time.time()

            if (epoch + 1) % val_frequency == 0:
                self.validate()

    def validate(self):
        results = {"preds": [], "labels": []}
        total_loss = 0
        self.model.eval()

        with torch.no_grad():
            for batch, labels in self.val_loader:
                batch = batch.to(self.device)
                labels = labels.to(self.device)
                logits = self.model(batch)
                loss = self.criterion(logits, labels)
                total_loss += loss.item()
                preds = logits.argmax(dim=-1).cpu().numpy()
                results["preds"].extend(list(preds))
                results["labels"].extend(list(labels.cpu().numpy()))

        accuracy = compute_accuracy(
            np.array(results["labels"]), np.array(results["preds"])
        )
        average_loss = total_loss / len(self.val_loader)

        self.summary_writer.add_scalars("accuracy", {"test": accuracy}, self.step)
        self.summary_writer.add_scalars("loss", {"test": average_loss}, self.step)
        print(f"validation loss: {average_loss:.5f}, accuracy: {accuracy * 100:2.2f}")

def compute_accuracy(
    labels: Union[torch.Tensor, np.ndarray], preds: Union[torch.Tensor, np.ndarray]
) -> float:
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    assert len(labels) == len(preds)
    return float((labels == preds).sum()) / len(labels)

def get_summary_writer_log_dir(args: argparse.Namespace) -> str:
    # --- MOMENTUM TASK: Adjusted prefix string ---
    tb_log_dir_prefix = f'CNN_bn_bs={args.batch_size}_lr={args.learning_rate}_momentum=0.9_run_'
    i = 0
    while i < 1000:
        tb_log_dir = args.log_dir / (tb_log_dir_prefix + str(i))
        if not tb_log_dir.exists():
            return str(tb_log_dir)
        i += 1
    return str(tb_log_dir)

def main(args):
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
        print("CUDA not available, running on CPU instead.")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root=args.dataset_root, train=True, download=True, transform=transform
    )
    val_dataset = torchvision.datasets.CIFAR10(
        root=args.dataset_root, train=False, download=True, transform=transform
    )

    train_loader = DataLoader(
        train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.worker_count
    )
    val_loader = DataLoader(
        val_dataset, batch_size=args.batch_size, shuffle=False, num_workers=args.worker_count
    )

    model = CNN()
    criterion = nn.CrossEntropyLoss()

    # --- MOMENTUM TASK: Initialise SGD optimizer with momentum=0.9 ---
    optimizer = torch.optim.SGD(model.parameters(), lr=args.learning_rate, momentum=0.9)

    log_dir = get_summary_writer_log_dir(args)
    summary_writer = SummaryWriter(log_dir=log_dir)

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        summary_writer=summary_writer,
        device=device,
    )

    trainer.train(
        epochs=args.epochs,
        val_frequency=args.val_frequency,
        print_frequency=args.print_frequency,
        log_frequency=args.log_frequency,
    )

    summary_writer.close()

if __name__ == "__main__":
    import shutil
    from google.colab import drive
    from torchvision.datasets.utils import check_integrity, extract_archive

    drive.mount("/content/drive")
    args = parser.parse_args([])

    saved_archive = Path(
        "/content/drive/MyDrive/deep_learning_lab/datasets/"
        "cifar-10-python.tar.gz"
    )

    args.dataset_root = Path("/content/cifar10_data")
    args.dataset_root.mkdir(parents=True, exist_ok=True)

    dataset_class = torchvision.datasets.CIFAR10
    dataset_folder = args.dataset_root / dataset_class.base_folder

    required_files = (
        dataset_class.train_list
        + dataset_class.test_list
        + [(dataset_class.meta["filename"], dataset_class.meta["md5"])]
    )

    def local_dataset_is_valid():
        return all(
            check_integrity(str(dataset_folder / filename), checksum)
            for filename, checksum in required_files
        )

    if not local_dataset_is_valid():
        if not check_integrity(str(saved_archive), dataset_class.tgz_md5):
            raise RuntimeError(
                "The saved CIFAR-10 archive is missing or corrupted. "
                "Check My Drive/deep_learning_lab/datasets/"
                "cifar-10-python.tar.gz. No download was attempted."
            )

        local_archive = args.dataset_root / dataset_class.filename

        print("Copying saved CIFAR-10 from Google Drive...")
        shutil.copy2(saved_archive, local_archive)

        if not check_integrity(str(local_archive), dataset_class.tgz_md5):
            raise RuntimeError("The local archive copy failed verification.")

        print("Extracting locally...")
        extract_archive(str(local_archive), str(args.dataset_root))

        if not local_dataset_is_valid():
            raise RuntimeError("Extracted dataset failed verification.")

    print("Using verified CIFAR-10 files — no internet download needed.")

    args.log_dir = Path("/content/drive/MyDrive/deep_learning_lab/logs")
    args.log_dir.mkdir(parents=True, exist_ok=True)
    args.worker_count = 2

    main(args)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using verified CIFAR-10 files — no internet download needed.
epoch: [0], step: [10], batch loss: 2.35233, batch accuracy: 33.59, data load time: 0.00708, step time: 0.01085
epoch: [0], step: [20], batch loss: 3.02072, batch accuracy: 32.03, data load time: 0.00575, step time: 0.00875
epoch: [0], step: [30], batch loss: 2.31255, batch accuracy: 34.38, data load time: 0.00523, step time: 0.00695
epoch: [0], step: [40], batch loss: 2.21632, batch accuracy: 39.06, data load time: 0.00709, step time: 0.01099
epoch: [0], step: [50], batch loss: 2.32396, batch accuracy: 34.38, data load time: 0.00597, step time: 0.00692
epoch: [0], step: [60], batch loss: 1.76651, batch accuracy: 44.53, data load time: 0.00496, step time: 0.01044
epoch: [0], step: [70], batch loss: 1.72864, batch accuracy: 49.22, data load time: 0.00705, step time: 0.01393
epoch: [0], step: [80], ba


# End of Lab 3

---
# Addendum 1: A note on Batch Normalisation

Whilst batch normalisation was originally motivated by the desire to reduce internal covariate shift, a [recent analysis](https://papers.nips.cc/paper/7996-understanding-batch-normalization.pdf) of the technique suggest that the key benefit of the technique is to constrain activations to zero mean and unit variance. Without this constraint, very deep networks (e.g. 110 layer ResNets) suffer from increasingly large activations with depth in the network. These large activations cause the network to diverge (the loss increases without bound) during training. Consequently deep networks without batch normalisation have to be trained with smaller learning rates than their batch normalised equivalents. The larger learning rates help reduce over fitting of the network, speed up convergence, and tend to improve the final accuracy too!
If you're interested in learning more, read [Understanding Batch Normalization - Bjorck et al (NeurIPS 2018)](https://papers.nips.cc/paper/7996-understanding-batch-normalization.pdf).

# Addendum 2: Learning Rate Decay

When you read deep learning related papers, often you will encounter figures with loss curve / accuracy like following: ![lr-decay-example](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/lr-decay.png?raw=1)

where you witness a jump in the training error, this is the result of applying learning rate decay. (Source: the [ResNet](https://arxiv.org/pdf/1512.03385.pdf) paper.)

Scheduling learning rate is a empirical technique to ameliorate deep neural network training. Again, think of the surface of the loss function we're optimising as a landscape, the initial learning rate guides the physical ball, i.e. network parameters, rolling downhill quickly; as descending goes on, the physical ball may oscillate around a local minimum if the learning rate is too big. A decayed learning rate in this stage can hence help the network converge to a local minimum.

Pytorch support multiple learning rate schedulers, here we will apply the Step scheduler to our training.

**Task**:
- After defining the `optimizer`, define a [`StepLR`](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.StepLR.html), which scales the learning rate by 0.1 every 20 epochs:
```python
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)
```
- Add `scheduler` argument to `Trainer.__init__` and pass the `scheduler` to it in `Training` instantiation.
- At the end of each training epoch, call `self.scheduler.step()`, which increment the scheduler counter by 1 epoch. The scheduler will update the learning rate of the next epoch when necessary.
- Change `tb_log_dir_prefix` in `get_summary_writer_log_dir` accordingly.
- Before, we train the network for 20 epochs; now, train the network for 40 epochs, with BS=128, LR=1e-1. As a result of using `stepLR`, in the first 20 epoch lr=0.1, and in the next 20 epochs lr=0.01. Expect to get an accuracy of ~78%.

Verify the effect of step scheduler by inspecting the loss curve and the accuracy.

![loss_stepLR](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/loss-stepLR.png?raw=1)
![accuracy_stepLR](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/accuracy-stepLR.png?raw=1)

## Reading
- [HOW DOES LEARNING RATE DECAY HELP MODERN NEURAL NETWORKS?](https://openreview.net/pdf?id=r1eOnh4YPB#:~:text=Learning%20rate%20decay%20(lrDecay)%20is,help%20both%20optimization%20and%20generalization.)
- [Learing Rate Wikipedia](https://en.wikipedia.org/wiki/Learning_rate)
- [Optimisation Chapter in the Deep Learning book](https://www.deeplearningbook.org/contents/optimization.html)

**Question** what are other methods you could consider to stabilise the gradients during training?

# Addendum 3: CHALLENGE! Implementing BatchNorm2d

Read the Section 3 of the [BatchNorm](https://arxiv.org/pdf/1502.03167.pdf) paper, implement BatchNorm2d following _Algorithm 1_ and _Algorithm 2_. Track the dataset statistic using running average: $$ \hat{x}_{new} = (1 - \text{momentum}) * \hat{x} + \text{momentum} * x_t$$ where $\hat{x}$ is the estimated statistic and $x_t$ is the new observed value.

By default set momentum=0.1 and $\epsilon$=1e-5.

Code skeleton:

```python
import torch.nn as nn

class MyBatchNorm2d(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        # TASK: Define and initialise parameters
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            # TASK: Implement training phase
            pass
        else:
            # TASK: Implement testing phase
            pass
        raise NotImplementedError
```

Q: Which dimensions to average over? _Hint: BatchNorm2d is also called SpatialBatchNorm. Read Sec 3.2_  
Q: Which parameters are trainable? Which parameters are calculate directly / not trainable? *Hint: [nn.Parameter](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html#torch.nn.parameter.Parameter) vs [self.register\_buffer](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer)*. _Hint2: [discussion](https://stackoverflow.com/questions/57540745/what-is-the-difference-between-register-parameter-and-register-buffer-in-pytorch)._  
Q: How to initialize affine parameters correctly? _Hint: Initially BN acts as a plain normalising function_  
Q: How to initialize dataset statistic correctly? _Hint: Normalised values are assumed to be $N(0, I)$_  

**Note**
- Each object of `torch.nn.Module` class has a property called `self.training`, which indicates whether the object (model) is in training mode or not. Setting `model.eval()` or `model.train()` will update this property.

You should be able get the same performance using your own BatchNorm.

![Your-Own-BN](https://github.com/COMS30117-Deep-Learning/labsheets/blob/main/lab-3-training/media/your-own-bn.png?raw=1)

*We provide our implementation of BatchNorm in `lab-3-training/code/batch_norm_ref.py` for your reference, don't look at it if you want to explore yourself!*